# AIC2026 TEAM-EVAL E0/E1 — Cross-Level Bootstrap

Team-neutral source-pool preparation only. This notebook does not create semantic queries or ground truth.

Required Kaggle inputs:
1. Raw AIC corpus: `/kaggle/input/datasets/nadkli/dataset-aic` (nested roots supported).
2. Optional L21 dataset containing `aic2026_l21_eval_bootstrap.zip`, default `/kaggle/input/datasets/irthn1311/aic2026-l21-eval-bootstrap`. Absence safely produces `SKIPPED_NO_INPUT`.

Internet required: **Yes, only for cloning the `TRIAGEEG` repository branch**. Corpus processing, atlas rendering and evaluation use no network or model download. Output ZIP: `/kaggle/working/aic2026_team_eval_e01_bundle.zip`.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZipFile

DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
L21_INPUT = Path(os.environ.get('AIC_L21_BOOTSTRAP_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-l21-eval-bootstrap'))
OUTPUT_ROOT = Path('/kaggle/working/aic2026_team_eval_e01')
ZIP_PATH = Path('/kaggle/working/aic2026_team_eval_e01_bundle.zip')
print({'repo_url': REPO_URL, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR), 'data_input': str(DATA_INPUT), 'optional_l21_input': str(L21_INPUT), 'output_zip': str(ZIP_PATH), 'internet_required_for_repo_clone': True, 'model_download_required': False})

In [ ]:
if REFRESH_REPO and REPO_DIR.exists():
    if REPO_DIR.parent != Path('/kaggle/working') or REPO_DIR.name != 'AIC2026_TeamPTK_SGU': raise RuntimeError(f'Refusing repository cleanup outside the expected Kaggle path: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'src/aic2026_eval/pipeline.py').is_file(): raise RuntimeError(f'Cloned repository/ref does not contain TEAM-EVAL package: {REPO_DIR}; push the current TRIAGEEG changes first')
REPO_ROOT = REPO_DIR.resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file
DATASET_ROOT = resolve_dataset_root(DATA_INPUT)
L21_ZIP = resolve_named_file(L21_INPUT, 'aic2026_l21_eval_bootstrap.zip', optional=True)
BUILD_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print({'resolved_repo': str(REPO_ROOT), 'resolved_dataset': str(DATASET_ROOT), 'resolved_l21_zip': str(L21_ZIP) if L21_ZIP else None, 'commit': BUILD_COMMIT})

In [ ]:
from aic2026_eval.pipeline import run_bootstrap

RESULT = run_bootstrap(dataset_root=DATASET_ROOT, repository_root=REPO_ROOT, output_root=OUTPUT_ROOT, l21_bootstrap_zip=L21_ZIP, manual_exclude_path=REPO_ROOT / 'configs/eval/manual_exclude_videos.txt', build_commit=BUILD_COMMIT)
print(json.dumps({key: value for key, value in RESULT.items() if key.isupper()}, indent=2))

In [ ]:
CORPUS = RESULT['corpus_summary']
assert CORPUS['status'] == 'PASS' and CORPUS['video_count'] > 0
print('Corpus by source group:', json.dumps(CORPUS['by_source_group'], indent=2))

In [ ]:
USAGE = RESULT['usage_summary']
assert USAGE['status'] == 'PASS'
print('Usage tiers:', json.dumps(USAGE['by_usage_tier'], indent=2), 'scan_mode:', USAGE['repository_scan_mode'])

In [ ]:
print('L21 mapping audit:', json.dumps(RESULT['l21_mapping_summary'], indent=2))
assert RESULT['L21_MAPPING_AUDIT'] in {'PASS', 'PARTIAL', 'SKIPPED'}

In [ ]:
SELECTION, ATLAS = RESULT['selection_report'], RESULT['atlas_index']
assert SELECTION['candidate_count'] == 36
assert SELECTION['blind_candidate_count'] == 24 and SELECTION['sealed_candidate_count'] == 12
assert SELECTION['blind_sealed_video_overlap'] == 0 and SELECTION['t2_t3_used'] is False
assert ATLAS['status'] == 'READY' and ATLAS['atlas_sheet_count'] == 36
print(json.dumps({'selection': SELECTION, 'atlas': ATLAS}, indent=2))

In [ ]:
assert Path(RESULT['zip_path']) == ZIP_PATH and ZIP_PATH.is_file()
with ZipFile(ZIP_PATH) as archive:
    members = archive.namelist()
assert not any(name.endswith(('.mp4', '.npy', '.npz', '.pt', '.pth')) for name in members)
for key in ('TEAM_EVAL_INFRA_STATUS','CORPUS_INVENTORY','CONTAMINATION_CENSUS','L21_MAPPING_AUDIT','HELDOUT_CANDIDATES','BLIND_CANDIDATE_VIDEOS','SEALED_CANDIDATE_VIDEOS','BLIND_SEALED_VIDEO_OVERLAP','ATLAS_STATUS','DENSE_RENDERER_STATUS','PREDICTION_VALIDATOR','SHARED_EVALUATOR','READY_FOR_AI_SEMANTIC_SELECTION'):
    print(f'{key}={RESULT[key]}')
print('BLIND_BENCHMARK_COMPLETE=NO')
print('DOWNLOAD ZIP:', ZIP_PATH, 'size_bytes=', ZIP_PATH.stat().st_size, 'members=', len(members))